# 🏒 UniVision2Board — YOLOv11 Training auf Google Colab

**Modell: `yolo11m.pt`** — 20M Parameter, 8× mehr Kapazität als das bisherige Nano-Modell.
Erkennt auf demselben Frame bereits **14 Personen ohne Training** (COCO-Vorwissen).
Nach Fine-Tuning auf unserem Datensatz: erwartete Verbesserung von mAP50 ~0.43 → **0.55–0.65**.

**Workflow:**
1. GPU prüfen (T4 hat 16 GB VRAM — reicht problemlos für yolo11m)
2. Google Drive verbinden
3. Datensatz entpacken
4. Training starten (`yolo11m`, 150 Epochen, ~25–35 Min. auf T4)
5. Bestes Modell zurück nach Drive speichern

**Voraussetzung:** `colab_dataset.zip` liegt in deinem Google Drive unter `Mein Drive/KIP_Training/`

## Schritt 1 — GPU prüfen

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}  |  VRAM: {vram:.1f} GB')
else:
    print('❌ Keine GPU — gehe zu Laufzeit > Laufzeittyp ändern > T4 GPU')

## Schritt 2 — Abhängigkeiten installieren

In [ ]:
!pip install ultralytics -q
print('✅ ultralytics installiert')

## Schritt 3 — Google Drive verbinden & Datensatz entpacken

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
from pathlib import Path

# ── Pfade anpassen falls nötig ────────────────────────────────────────────
DRIVE_ZIP  = Path('/content/drive/MyDrive/KIP_Training/colab_dataset.zip')
WORK_DIR   = Path('/content/kip')
DRIVE_OUT  = Path('/content/drive/MyDrive/KIP_Training/models')
# ─────────────────────────────────────────────────────────────────────────

DRIVE_OUT.mkdir(parents=True, exist_ok=True)

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(
        f'ZIP nicht gefunden: {DRIVE_ZIP}\n'
        'Bitte colab_dataset.zip in Google Drive/KIP_Training/ hochladen.'
    )

print(f'Entpacke {DRIVE_ZIP.name} ({DRIVE_ZIP.stat().st_size/1e6:.0f} MB)...')
WORK_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP) as zf:
    zf.extractall(WORK_DIR)

# Bilder und Labels zählen
ds = WORK_DIR / 'dataset'
n_train = len(list((ds/'images'/'train').glob('*')))
n_val   = len(list((ds/'images'/'val').glob('*')))
print(f'✅ Datensatz: {n_train} Train-Bilder, {n_val} Val-Bilder')

## Schritt 4 — data.yaml für Colab anpassen

In [ ]:
import yaml

yaml_path = WORK_DIR / 'data.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

# Pfad auf Colab-Verzeichnis setzen
cfg['path'] = str(ds)

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)

print(f'✅ data.yaml angepasst:')
print(f'   path:  {cfg["path"]}')
print(f'   train: {cfg["train"]}')
print(f'   val:   {cfg["val"]}')
print(f'   Klassen: {cfg["names"]}')

## Schritt 5 — Training starten

**Modell-Optionen (Stand: yolo11m bereits trainiert):**
| Modell | Params | Colab-Zeit | Tipp |
|--------|--------|------------|------|
| `yolo11m.pt` | 20M | ~30 Min | Bereits trainiert |
| **`yolo11x.pt`** | **56M** | **~55 Min** | **Nächste Stufe** |
| `yolo11m.pt` + imgsz=1280 | 20M | ~50 Min | **Besser für kleine Spieler** |

> **Tipp:** Für weit entfernte / kleine Spieler bringt `imgsz=1280` oft mehr als das nächstgrössere Modell.
> Setze `STRATEGY = 'yolo11x'` für grösseres Modell oder `STRATEGY = 'hires'` für höhere Auflösung.


In [ ]:
from ultralytics import YOLO
import torch

# ══════════════════════════════════════════════════════════════════════════
# STRATEGIE WÄHLEN:
#   'yolo11x'  → grösseres Modell (56M Params, ~55 Min auf T4)
#   'hires'    → yolo11m mit imgsz=1280 (besser für kleine/weit entfernte Spieler)
STRATEGY = 'yolo11x'   # ← hier ändern
# ══════════════════════════════════════════════════════════════════════════

EPOCHS   = 150
PATIENCE = 30
DEVICE   = 0

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU-VRAM: {vram_gb:.1f} GB')

if STRATEGY == 'yolo11x':
    BASE_MODEL = 'yolo11x.pt'
    IMGSZ      = 640
    BATCH      = 8 if vram_gb < 20 else 16
elif STRATEGY == 'hires':
    BASE_MODEL = 'yolo11m.pt'
    IMGSZ      = 1280
    BATCH      = 8 if vram_gb < 20 else 12
else:
    raise ValueError(f'Unbekannte Strategie: {STRATEGY}')

print(f'Strategie: {STRATEGY}  |  Modell: {BASE_MODEL}  |  imgsz: {IMGSZ}  |  Batch: {BATCH}')
print()

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    patience=PATIENCE,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    project='/content/runs',
    name='kip_train',
    exist_ok=True,
    verbose=True,
)

## Schritt 6 — Ergebnisse anzeigen & Modell nach Drive speichern

In [ ]:
import shutil
from pathlib import Path

best_pt   = Path('/content/runs/kip_train/weights/best.pt')
last_pt   = Path('/content/runs/kip_train/weights/last.pt')
result_csv = Path('/content/runs/kip_train/results.csv')

if best_pt.exists():
    dst_best = DRIVE_OUT / 'best.pt'
    dst_last = DRIVE_OUT / 'last.pt'
    shutil.copy(best_pt, dst_best)
    shutil.copy(last_pt, dst_last)
    if result_csv.exists():
        shutil.copy(result_csv, DRIVE_OUT / 'results.csv')
    print(f'✅ Modell gespeichert:')
    print(f'   {dst_best}  ({dst_best.stat().st_size/1e6:.1f} MB)')
    print()
    print('Jetzt best.pt aus Google Drive herunterladen und nach')
    print('  KIP/finetune/runs/train/weights/best.pt  kopieren.')
else:
    print('❌ best.pt nicht gefunden — Training abgebrochen?')

## Schritt 6b — Klassen-Ergebnisse im Detail

In [ ]:
import pandas as pd
from pathlib import Path

result_csv = Path('/content/runs/kip_train/results.csv')
best_pt    = Path('/content/runs/kip_train/weights/best.pt')

if result_csv.exists():
    df = pd.read_csv(result_csv)
    df.columns = df.columns.str.strip()
    best_idx = df['metrics/mAP50(B)'].idxmax()
    best = df.loc[best_idx]
    print('╔══════════════════════════════════╗')
    print('║       TRAININGS-ERGEBNIS         ║')
    print('╠══════════════════════════════════╣')
    print(f'║  Beste Epoche : {int(best["epoch"]):>3}              ║')
    print(f'║  mAP50        : {best["metrics/mAP50(B)"]:.3f}            ║')
    print(f'║  mAP50-95     : {best["metrics/mAP50-95(B)"]:.3f}            ║')
    print(f'║  Precision    : {best["metrics/precision(B)"]:.3f}            ║')
    print(f'║  Recall       : {best["metrics/recall(B)"]:.3f}            ║')
    print('╚══════════════════════════════════╝')

    # Per-Klasse
    print()
    model = YOLO(str(best_pt))
    val_r = model.val(data=str(yaml_path), device=DEVICE, verbose=False)
    names = ['player', 'goalkeeper', 'ball', 'referee']
    print('Klasse       mAP50   Recall  Precision')
    print('─────────────────────────────────────')
    for i, name in enumerate(names):
        try:
            m50 = float(val_r.box.maps[i])
            r   = float(val_r.box.r[i])
            p   = float(val_r.box.p[i])
            print(f'{name:<12} {m50:.3f}   {r:.3f}   {p:.3f}')
        except:
            pass

## Schritt 7 — Validierungsergebnisse

In [ ]:
from ultralytics import YOLO
import pandas as pd

# Ergebnisse aus CSV
if result_csv.exists():
    df = pd.read_csv(result_csv)
    df.columns = df.columns.str.strip()
    best_row = df.loc[df['metrics/mAP50(B)'].idxmax()]
    print('=== Beste Epoche ===')
    print(f'  Epoche:   {int(best_row["epoch"])}')
    print(f'  mAP50:    {best_row["metrics/mAP50(B)"]:.3f}')
    print(f'  mAP50-95: {best_row["metrics/mAP50-95(B)"]:.3f}')
    print(f'  Precision:{best_row["metrics/precision(B)"]:.3f}')
    print(f'  Recall:   {best_row["metrics/recall(B)"]:.3f}')

# Detaillierte Klassen-Validierung
print()
print('=== Klassen-Auswertung ===')
model = YOLO('/content/runs/kip_train/weights/best.pt')
val_results = model.val(data=str(yaml_path), device=DEVICE)
print(val_results)